# Checkpoint 4: Xử Lý Mất Cân Bằng Lớp Bằng Weighted Cross-Entropy Loss (Class Weights)

Nghiên cứu: Nâng cao hiệu năng mô hình phát hiện phát ngôn thù ghét trên mạng xã hội tiếng Việt (ViHSD).

Mục tiêu kỹ thuật của Checkpoint 4:
1. Giải quyết vấn đề mất cân bằng lớp nghiêm trọng trong tập dữ liệu ViHSD: Lớp 0 (CLEAN ~82.7%), Lớp 1 (OFFENSIVE ~6.7%), Lớp 2 (HATE ~10.6%).
2. Áp dụng cơ chế phạt tĩnh (Static Loss Weighting) thông qua hàm mất mát Weighted Cross-Entropy Loss:
   $$\mathcal{L}_{WCE}(z, y) = -w_y \log \left( \frac{\exp(z_y)}{\sum_{j=0}^{K-1} \exp(z_j)} \right)$$
   Trong đó $w_y$ là trọng số phân bổ lớp gán cho nhãn thực tế $y$, buộc mô hình phải cập nhật gradient mạnh mẽ hơn khi dự đoán sai các mẫu thuộc nhãn thiểu số (OFFENSIVE và HATE).
3. Khảo sát và so sánh 4 phương pháp toán học tính toán vector trọng số lớp:
   - Nghịch đảo tần suất lớp (Inverse Class Frequency - ICF): $w_c = \frac{N}{K \cdot N_c}$ (Chuẩn Scikit-learn).
   - Làm mịn căn bậc hai (Smoothed Inverse Frequency): $w_c = \sqrt{\frac{N}{N_c}}$, chuẩn hóa $\text{mean}(w) = 1.0$.
   - Số lượng mẫu hiệu dụng (Effective Number of Samples - Cui et al., 2019): $w_c \propto \frac{1 - \beta}{1 - \beta^{N_c}}$ với $\beta = 0.999$.
   - Trọng số lũy thừa làm dịu (Mild Power-Law Class Weights): $w_c = \left( \frac{N}{N_c} \right)^\rho$ với $\rho = 0.25$, chuẩn hóa $\text{mean}(w) = 1.0$.
4. Phân tích sự đánh đổi giữa độ nhạy (Recall) của các lớp thiểu số và độ chuẩn xác (Precision) của lớp đa số CLEAN.
5. Kế thừa toàn bộ chiến lược huấn luyện tối ưu đã thiết lập từ Checkpoint 2: mở rộng ngữ cảnh max_length = 256, phân tầng tốc độ học (Differential Learning Rates: Backbone 2e-5, Classification Head 1e-4), lịch trình Cosine Annealing kèm Warmup, và lưu trữ trọng số tối ưu nhất theo eval_macro_f1 trên tập dev với Early Stopping.

Ghi chú: Toàn bộ mã nguồn tiền xử lý, cấu trúc Dataset, thuật toán tính trọng số, lớp tùy biến WeightedCrossEntropyTrainer, các hàm đánh giá và trực quan hóa được tích hợp trực tiếp 100% trong notebook nhằm đảm bảo tính độc lập hoàn toàn khi thực thi trên Google Colab.

In [ ]:
import os
import sys

# Thiết lập môi trường làm việc và kết nối Google Drive trên Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    project_dir = '/content/drive/MyDrive/NLP_PhoBert_ViHSD'
except ImportError:
    project_dir = os.path.abspath('..')

checkpoints_dir = os.path.join(project_dir, 'checkpoints', 'checkpoint04_power')
outputs_dir = os.path.join(project_dir, 'outputs', 'checkpoint04_power')

os.makedirs(checkpoints_dir, exist_ok=True)
os.makedirs(outputs_dir, exist_ok=True)

print(f'Thư mục dự án: {project_dir}')
print(f'Thư mục lưu checkpoint: {checkpoints_dir}')
print(f'Thư mục lưu kết quả: {outputs_dir}')

In [ ]:
# Cài đặt các thư viện cần thiết trên môi trường Google Colab
!pip install -q pyvi transformers accelerate scikit-learn seaborn matplotlib pandas numpy torch

In [ ]:
# Thiết lập hạt giống ngẫu nhiên cố định để đảm bảo tính tái lập thực nghiệm
random_seed = 42

import os
import random
import numpy as np
import torch

random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f'Hạt giống ngẫu nhiên đã được cố định: {random_seed}')
print(f'Thiết bị tính toán: {"CUDA (GPU)" if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    print(f'Tên GPU: {torch.cuda.get_device_name(0)}')
    print(f'Dung lượng VRAM khả dụng: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')

## 1. Nạp Tập Dữ Liệu Thô và Kiểm Tra Trước Xử Lý (Pre-pipeline Verification)

Kiểm tra tính toàn vẹn của ba tập dữ liệu chuẩn: `train.csv` (24,048 mẫu), `dev.csv` (2,672 mẫu) và `test.csv` (6,680 mẫu). Đảm bảo số lượng mẫu khớp tuyệt đối với nghiên cứu cơ sở của Luu et al. (2021) và phân tích sự phân bổ mất cân bằng nghiêm trọng giữa các nhãn.

In [ ]:
# Cấu hình đường dẫn tập dữ liệu
data_dir = 'data' if os.path.exists('data/train.csv') else os.path.join(project_dir, 'data')
train_path = os.path.join(data_dir, 'train.csv')
dev_path = os.path.join(data_dir, 'dev.csv')
test_path = os.path.join(data_dir, 'test.csv')

import pandas as pd

print(f'Đọc tập Train từ: {train_path}')
print(f'Đọc tập Dev từ:   {dev_path}')
print(f'Đọc tập Test từ:  {test_path}')

train_raw = pd.read_csv(train_path)
dev_raw = pd.read_csv(dev_path)
test_raw = pd.read_csv(test_path)

# Kiểm tra toàn vẹn số lượng mẫu và giá trị thiếu trước khi đưa vào pipeline
assert len(train_raw) == 24048, f'Số lượng mẫu train không khớp: {len(train_raw)}'
assert len(dev_raw) == 2672, f'Số lượng mẫu dev không khớp: {len(dev_raw)}'
assert len(test_raw) == 6680, f'Số lượng mẫu test không khớp: {len(test_raw)}'

print('Kiểm tra tập dữ liệu thô trước xử lý:')
print(f'- Train: {len(train_raw)} mẫu, giá trị thiếu free_text: {train_raw["free_text"].isna().sum()}')
print(f'- Dev:   {len(dev_raw)} mẫu, giá trị thiếu free_text: {dev_raw["free_text"].isna().sum()}')
print(f'- Test:  {len(test_raw)} mẫu, giá trị thiếu free_text: {test_raw["free_text"].isna().sum()}')

# Thống kê phân bổ nhãn trên tập huấn luyện
train_dist = train_raw['label_id'].value_counts().sort_index()
print('\nPhân bổ nhãn trên tập huấn luyện:')
class_labels = {0: 'CLEAN', 1: 'OFFENSIVE', 2: 'HATE'}
for label_id, count in train_dist.items():
    pct = (count / len(train_raw)) * 100
    print(f'  Lớp {label_id} ({class_labels.get(label_id, "UNKNOWN")}): {count:5d} mẫu ({pct:5.2f}%)')

## 2. Định Nghĩa Trực Tiếp Lớp Tiền Xử Lý ViHSDPreprocessor

Tái sử dụng quy trình tiền xử lý chuẩn hóa đã kiểm chứng ở Checkpoint 1:
1. Làm sạch URL, Email, thẻ HTML và chuẩn hóa khoảng trắng thừa.
2. Xử lý biểu tượng cảm xúc (Emoji) 2 pha: đệm khoảng trắng bảo vệ từ tố và ánh xạ sang từ vựng tiếng Việt tương ứng.
3. Chuẩn hóa teencode, từ lóng và biến thể né kiểm duyệt bằng ranh giới từ (Word Boundary Protection).
4. Thu gọn ký tự lặp kéo dài (Lengthening Reduction) về 1 ký tự duy nhất.
5. Phân đoạn từ tiếng Việt tự động bằng thư viện `PyVi` theo chuẩn đầu vào PhoBERT.
6. Bảo tồn toàn bộ đại từ nhân xưng (`mày`, `tao`, `nó`, `thằng`, `con`) và từ phủ định (`không`, `chẳng`, `chả`).

In [ ]:
import os
import re
import pandas as pd
import numpy as np

try:
    from pyvi import ViTokenizer
except ImportError:
    ViTokenizer = None


class ViHSDPreprocessor:
    """Bộ tiền xử lý và chuẩn hóa văn bản cho bài toán phát hiện phát ngôn thù ghét (ViHSD).

    Quy trình xử lý bao gồm:
    1. Làm sạch URL, Email, thẻ HTML và chuẩn hóa khoảng trắng.
    2. Bảo tồn và ánh xạ sắc thái cảm xúc (Emoji) qua quy trình 2 pha dựa vào từ điển bên ngoài.
    3. Chuẩn hóa teencode, tiếng lóng và biến thể né kiểm duyệt qua từ điển bên ngoài.
    4. Thu gọn ký tự lặp kéo dài (Lengthening Reduction).
    5. Bảo tồn nguyên vẹn từ dừng, đại từ nhân xưng và từ phủ định.
    6. Phân đoạn từ tiếng Việt sử dụng PyVi.
    """

    # Khởi tạo đối tượng tiền xử lý với các đường dẫn từ điển teencode và emoji
    def __init__(self, teencode_path='data/teencode.csv', emoji_path='data/emojis.csv'):
        self.teencode_dict = self.load_teencode(teencode_path)
        self.emoji_dict = self.load_emoji_mapping(emoji_path)
        self._init_regex_patterns()

    # Biên dịch trước toàn bộ biểu thức chính quy nhằm tối ưu tốc độ xử lý trên tập dữ liệu lớn
    def _init_regex_patterns(self):
        self.url_pattern = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
        self.email_pattern = re.compile(r'\S+@\S+', re.IGNORECASE)
        self.html_pattern = re.compile(r'<.*?>')
        self.whitespace_pattern = re.compile(r'\s+')

        # Nhận diện ký tự chữ cái lặp lại từ 3 lần trở lên bao gồm cả bảng chữ cái tiếng Việt
        self.lengthening_pattern = re.compile(
            r'([a-zA-ZàáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđĐ])\1{2,}',
            re.IGNORECASE,
        )

        # Biểu thức đệm khoảng trắng bao quanh emoji để ngăn lỗi dính từ khi tách từ bằng PyVi
        self.emoji_padding_pattern = re.compile(
            r'(\s*)([\U00010000-\U0010ffff\u2600-\u27ff])(\s*)'
        )

        # Mẫu khớp từ điển emoji theo thứ tự độ dài giảm dần
        sorted_emojis = sorted(self.emoji_dict.keys(), key=len, reverse=True)
        self.emoji_replace_pattern = re.compile(
            '|'.join(re.escape(k) for k in sorted_emojis)
        )

        # Mẫu khớp teencode kết hợp biên giới từ để tránh thay thế nhầm từ tố hợp lệ
        sorted_teencode = sorted(self.teencode_dict.keys(), key=len, reverse=True)
        teencode_patterns = []
        for k in sorted_teencode:
            escaped = re.escape(k)
            left = r'(?<!\w)' if re.match(r'^\w', k) else ''
            right = r'(?!\w)' if re.search(r'\w$', k) else ''
            teencode_patterns.append(f'{left}{escaped}{right}')
        self.teencode_pattern = re.compile(
            '|'.join(teencode_patterns), re.IGNORECASE
        )

    # Đọc từ điển teencode và từ lóng từ tệp CSV thành từ điển ánh xạ
    def load_teencode(self, path):
        df = pd.read_csv(path)
        return dict(
            zip(
                df['teencode'].astype(str).str.strip().str.lower(),
                df['replacement'].astype(str).str.strip().str.lower(),
            )
        )

    # Đọc bảng ánh xạ biểu tượng cảm xúc (emoji) từ tệp CSV thành từ điển thay thế
    def load_emoji_mapping(self, path):
        df = pd.read_csv(path)
        return dict(
            zip(
                df['emoji'].astype(str).str.strip(),
                df['replacement'].astype(str).str.strip(),
            )
        )

    # Loại bỏ đường dẫn URL, địa chỉ email, thẻ HTML và khoảng trắng thừa
    def clean_special_characters(self, text):
        text = self.url_pattern.sub(' ', text)
        text = self.email_pattern.sub(' ', text)
        text = self.html_pattern.sub(' ', text)
        return self.whitespace_pattern.sub(' ', text).strip()

    # Xử lý emoji 2 pha: đệm khoảng trắng bảo vệ từ tố rồi ánh xạ ngữ nghĩa sang PhoBERT
    def pad_and_map_emojis(self, text):
        text = self.emoji_padding_pattern.sub(r' \2 ', text)
        text = self.emoji_replace_pattern.sub(
            lambda m: f' {self.emoji_dict[m.group(0)]} ', text
        )
        return self.whitespace_pattern.sub(' ', text).strip()

    # Chuẩn hóa từ viết tắt và tiếng lóng dựa trên biên giới từ
    def normalize_teencode(self, text):
        return self.teencode_pattern.sub(
            lambda m: self.teencode_dict.get(m.group(0).lower(), m.group(0)), text
        )

    # Thu gọn các ký tự chữ cái lặp lại liên tiếp từ 3 lần trở lên về 1 ký tự duy nhất
    def reduce_lengthening(self, text):
        return self.lengthening_pattern.sub(r'\1', text)

    # Phân đoạn từ tiếng Việt tự động bằng thư viện PyVi theo quy chuẩn PhoBERT
    def segment_words(self, text):
        if ViTokenizer is None:
            raise ImportError(
                'Thư viện pyvi chưa được cài đặt. Vui lòng cài đặt qua lệnh: pip install pyvi'
            )
        return ViTokenizer.tokenize(text)

    # Thực hiện quy trình tiền xử lý hoàn chỉnh cho một đoạn văn bản theo đúng thứ tự logic
    def transform_text(self, text, lowercased=False):
        if not isinstance(text, str):
            text = str(text) if text is not None else ''

        if not text.strip():
            return ''

        # Thực hiện tuần tự: Làm sạch -> Emoji -> Teencode -> Thu gọn ký tự -> Tách từ PyVi
        text = self.clean_special_characters(text)
        text = self.pad_and_map_emojis(text)
        text = self.normalize_teencode(text)
        text = self.reduce_lengthening(text)

        # Bảo tồn toàn bộ đại từ nhân xưng và từ phủ định (tuyệt đối không lọc từ dừng)
        if text.strip():
            text = self.segment_words(text)

        if lowercased:
            text = text.lower()

        return self.whitespace_pattern.sub(' ', text).strip()

    # Áp dụng chuẩn hóa trên toàn bộ DataFrame, loại bỏ dòng rỗng/thiếu và trả về mảng đặc trưng X và nhãn y
    def transform_dataset(self, df, text_col='free_text', label_col='label_id', lowercased=False):
        df_clean = df.dropna(subset=[text_col, label_col]).copy()
        df_clean[label_col] = df_clean[label_col].astype(int)

        transformed_texts = []
        valid_indices = []

        for idx, row in df_clean.iterrows():
            cleaned_text = self.transform_text(row[text_col], lowercased=lowercased)
            if cleaned_text:
                transformed_texts.append(cleaned_text)
                valid_indices.append(idx)

        filtered_df = df_clean.loc[valid_indices].copy()
        X = transformed_texts
        y = filtered_df[label_col].to_numpy()

        return X, y

## 3. Khởi Tạo Bộ Tiền Xử Lý và Chuẩn Hóa Dữ Liệu

Áp dụng quy trình tiền xử lý đồng bộ trên cả ba tập `train.csv`, `dev.csv` và `test.csv`. Kiểm tra tính toàn vẹn (Assertion Checks) sau khi làm sạch nhằm loại bỏ hoàn toàn các chuỗi rỗng và đảm bảo tính nhất quán về kích thước và nhãn phân loại.

In [ ]:
# Cấu hình đường dẫn từ điển chuẩn hóa
teencode_path = os.path.join(data_dir, 'teencode.csv')
emoji_path = os.path.join(data_dir, 'emojis.csv')

print(f'Nạp từ điển teencode từ: {teencode_path}')
print(f'Nạp từ điển emoji từ:    {emoji_path}')

# Khởi tạo bộ tiền xử lý với từ điển ngoài
preprocessor = ViHSDPreprocessor(teencode_path=teencode_path, emoji_path=emoji_path)

# Thực thi tiền xử lý trên toàn bộ tập dữ liệu
text_column = 'free_text'
label_column = 'label_id'
preserve_casing = False

train_X, train_y = preprocessor.transform_dataset(
    train_raw, text_col=text_column, label_col=label_column, lowercased=preserve_casing
)
dev_X, dev_y = preprocessor.transform_dataset(
    dev_raw, text_col=text_column, label_col=label_column, lowercased=preserve_casing
)
test_X, test_y = preprocessor.transform_dataset(
    test_raw, text_col=text_column, label_col=label_column, lowercased=preserve_casing
)

# Kiểm tra tính toàn vẹn sau xử lý (Post-pipeline Verification)
assert len(train_X) == len(train_y), 'Số lượng đặc trưng và nhãn train không khớp'
assert len(dev_X) == len(dev_y), 'Số lượng đặc trưng và nhãn dev không khớp'
assert len(test_X) == len(test_y), 'Số lượng đặc trưng và nhãn test không khớp'
assert sum(len(x.strip()) == 0 for x in train_X) == 0, 'Tồn tại chuỗi rỗng trong train_X'
assert set(np.unique(train_y)).issubset({0, 1, 2}), 'Tồn tại nhãn không hợp lệ trong train_y'

print('Kết quả tiền xử lý dữ liệu thành công:')
print(f'- Tập Train hợp lệ: {len(train_X)} / {len(train_raw)}')
print(f'- Tập Dev hợp lệ:   {len(dev_X)} / {len(dev_raw)}')
print(f'- Tập Test hợp lệ:  {len(test_X)} / {len(test_raw)}')

## 4. Định Nghĩa Lớp BuildDataset và Mã Hóa Với Độ Dài Mở Rộng (max_length = 256)

Kế thừa cấu hình tokenization tối ưu từ Checkpoint 2:
- Mở rộng độ dài chuỗi tối đa `max_length = 256` token, bao phủ 99.8% độ dài bình luận thực tế trên mạng xã hội mà không làm mất thông tin ngữ cảnh quan trọng.
- Sử dụng `AutoTokenizer.from_pretrained('vinai/phobert-base', use_fast=False)` tương thích với thuật toán FastBPE của PhoBERT.
- Đóng gói dữ liệu vào lớp `BuildDataset` kế thừa từ `torch.utils.data.Dataset`.

In [ ]:
# Cấu hình tham số mã hóa chuỗi mở rộng cho Checkpoint 4
pretrained_model_name = 'vinai/phobert-base'
max_seq_length = 256

import torch
from transformers import AutoTokenizer


class BuildDataset(torch.utils.data.Dataset):
    """Lớp quản lý và đóng gói tập dữ liệu tương thích với Hugging Face Trainer và mô hình PhoBERT."""

    # Khởi tạo đối tượng tập dữ liệu với đặc trưng token hóa và mảng nhãn tương ứng
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    # Trích xuất mẫu dữ liệu tại chỉ mục xác định và đóng gói thành PyTorch Tensor
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    # Trả về tổng số lượng mẫu quan sát có trong tập dữ liệu
    def __len__(self):
        return len(self.labels)


print(f'Khởi tạo Tokenizer: {pretrained_model_name} (use_fast=False, max_length={max_seq_length})')
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name, use_fast=False)

train_encodings = tokenizer(train_X, truncation=True, padding=True, max_length=max_seq_length)
dev_encodings = tokenizer(dev_X, truncation=True, padding=True, max_length=max_seq_length)
test_encodings = tokenizer(test_X, truncation=True, padding=True, max_length=max_seq_length)

train_dataset = BuildDataset(train_encodings, train_y)
dev_dataset = BuildDataset(dev_encodings, dev_y)
test_dataset = BuildDataset(test_encodings, test_y)

print(f'Kích thước tập train sau mã hóa: {len(train_dataset)}')
print(f'Kích thước tập dev sau mã hóa:   {len(dev_dataset)}')
print(f'Kích thước tập test sau mã hóa:  {len(test_dataset)}')

## 5. Tính Toán Trọng Số Lớp và Định Nghĩa Lớp WeightedCrossEntropyTrainer

Khảo sát 4 phương pháp toán học tính toán vector trọng số lớp trên tập huấn luyện `train_y`:
1. **Inverse Class Frequency (ICF - Scikit-learn):**
   $$w_c = \frac{N}{K \cdot N_c}$$
2. **Smoothed Inverse Frequency:**
   $$w_c = \sqrt{\frac{N}{N_c}} \bigg/ \left( \frac{1}{K} \sum_{j=0}^{K-1} \sqrt{\frac{N}{N_j}} \right)$$
3. **Effective Number of Samples (Cui et al., 2019):**
   $$E_c = \frac{1 - \beta^{N_c}}{1 - \beta}, \quad w_c \propto \frac{1}{E_c} \quad (\beta = 0.999)$$
4. **Mild Power-Law Class Weights (Lũy thừa làm dịu):**
   $$w_c = \left( \frac{N}{N_c} \right)^\rho \bigg/ \left( \frac{1}{K} \sum_{j=0}^{K-1} \left( \frac{N}{N_j} \right)^\rho \right) \quad (\rho = 0.25)$$
   *Ưu điểm:* Giảm tỷ lệ phạt giữa lớp thiểu số và đa số xuống mức vừa phải (~1.8x đến 2.0x), giúp kéo mạnh Recall mà không làm suy giảm nghiêm trọng Precision của lớp CLEAN.

Lớp tùy biến `WeightedCrossEntropyTrainer(Trainer)` ghi đè phương thức `compute_loss` để truyền vector trọng số `class_weights` vào `nn.CrossEntropyLoss(weight=...)`.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from transformers import Trainer


# Hàm tính toán vector trọng số phân bố lớp trên tập huấn luyện
def compute_class_weights_dict(labels, beta=0.999, power=0.25):
    """Tính toán vector trọng số phân bố lớp theo 4 phương pháp khác nhau."""
    labels = np.asarray(labels, dtype=int)
    total_samples = len(labels)
    classes, counts = np.unique(labels, return_counts=True)
    num_classes = len(classes)
    counts_float = counts.astype(np.float64)

    # 1. Phương pháp ICF (Inverse Class Frequency)
    w_icf = total_samples / (num_classes * counts_float)

    # 2. Phương pháp làm mịn căn bậc hai (Smoothed Inverse Frequency)
    raw_smoothed = np.sqrt(total_samples / counts_float)
    w_smoothed = raw_smoothed / np.mean(raw_smoothed)

    # 3. Phương pháp số lượng mẫu hiệu dụng (Effective Number of Samples)
    effective_num = (1.0 - np.power(beta, counts_float)) / (1.0 - beta)
    raw_eff = 1.0 / effective_num
    w_effective = (raw_eff / np.sum(raw_eff)) * num_classes

    # 4. Phương pháp Mild Power-Law (Lũy thừa làm dịu, rho = 0.25)
    raw_power = np.power(total_samples / counts_float, power)
    w_power = raw_power / np.mean(raw_power)

    return {
        'icf': torch.tensor(w_icf, dtype=torch.float32),
        'smoothed': torch.tensor(w_smoothed, dtype=torch.float32),
        'effective': torch.tensor(w_effective, dtype=torch.float32),
        'power': torch.tensor(w_power, dtype=torch.float32),
        'counts': dict(zip(classes, counts)),
    }


# Tính toán trọng số từ tập train_y thực tế
weights_catalog = compute_class_weights_dict(train_y, beta=0.999, power=0.25)
class_names_str = ['CLEAN', 'OFFENSIVE', 'HATE']

print('=' * 65)
print('BẢNG ĐỐI SÁNH VECTOR TRỌNG SỐ LỚP (CLASS WEIGHTS CATALOG)')
print('=' * 65)
for method_name, tensor_w in [('ICF (Chuẩn Scikit-learn)', weights_catalog['icf']),
                              ('Làm mịn căn bậc hai', weights_catalog['smoothed']),
                              ('Mẫu hiệu dụng (beta=0.999)', weights_catalog['effective']),
                              ('Mild Power-Law (rho=0.25)', weights_catalog['power'])]:
    w_list = [round(float(v), 4) for v in tensor_w]
    print(f'- {method_name:28s}: {w_list}')
print('=' * 65)

# Lựa chọn phương pháp Mild Power-Law (hoặc 'smoothed' / 'icf') làm cấu hình trọng số chính cho Checkpoint 4
selected_weight_method = 'power'
active_class_weights = weights_catalog[selected_weight_method]
print(f'Phương pháp trọng số được kích hoạt cho huấn luyện: {selected_weight_method.upper()}')
print(f'Vector trọng số áp dụng: {active_class_weights.tolist()}')


# Lớp Trainer tùy biến ghi đè compute_loss để truyền trọng số lớp vào Cross-Entropy Loss
class WeightedCrossEntropyTrainer(Trainer):
    """Lớp Trainer áp dụng vector trọng số phạt mất cân bằng lớp vào Cross-Entropy Loss."""

    # Khởi tạo Trainer với vector trọng số lớp
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    # Ghi đè phương thức tính toán tổn thất áp dụng trọng số lớp vào Cross-Entropy Loss
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        """Tính toán tổn thất có trọng số lớp trên mini-batch."""
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')

        weight = (
            self.class_weights.to(logits.device)
            if self.class_weights is not None
            else None
        )
        loss_fn = nn.CrossEntropyLoss(weight=weight)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


# Kiểm tra tính ổn định số học của hàm mất mát có trọng số
dummy_logits = torch.tensor([[2.0, 0.2, 0.1], [0.1, 2.5, 0.3]], dtype=torch.float32)
dummy_targets = torch.tensor([0, 1], dtype=torch.long)
test_loss = nn.CrossEntropyLoss(weight=active_class_weights)(dummy_logits, dummy_targets)
assert not torch.isnan(test_loss) and not torch.isinf(test_loss), 'Lỗi số học trong Weighted Cross-Entropy Loss'
print(f'Kiểm tra tính toán tổn thất Weighted Cross-Entropy thành công: {test_loss.item():.4f}')

## 6. Cấu Hình Siêu Tham Số, Phân Tầng Tốc Độ Học và Thiết Lập Trainer

Duy trì các thông số tối ưu từ Checkpoint 2 để đảm bảo so sánh công bằng:
- Tốc độ học phân tầng: $\text{lr}_{\text{backbone}} = 2 \times 10^{-5}$ cho PhoBERT và $\text{lr}_{\text{head}} = 1 \times 10^{-4}$ cho Classification Head.
- Tích lũy gradient: `per_device_train_batch_size = 16`, `gradient_accumulation_steps = 2` (kích thước batch hiệu dụng = 32).
- Lịch trình học: Cosine Annealing Scheduler kèm 10% Warmup tuyến tính qua 8 epochs.
- Chiến lược đánh giá và lưu checkpoint: `eval_strategy='epoch'`, `save_strategy='epoch'`, `load_best_model_at_end=True`, `metric_for_best_model='eval_macro_f1'`.
- Early Stopping: dừng sớm nếu chỉ số `eval_macro_f1` không cải thiện sau 2 epoch liên tiếp (`patience = 2`).

In [ ]:
# Thiết lập siêu tham số tối ưu hóa cho Checkpoint 4
num_labels = 3
num_epochs = 8
train_batch_size = 16
eval_batch_size = 16
gradient_accumulation_steps = 2
backbone_lr = 2e-5
head_lr = 1e-4
warmup_ratio = 0.1
weight_decay = 0.01
early_stopping_patience = 2

import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
    TrainingArguments,
)


# Hàm callback tính toán các chỉ số Accuracy, Precision, Recall và F1 trong quá trình huấn luyện
def compute_metrics_eval(eval_pred):
    """Hàm callback đánh giá tích hợp trực tiếp vào Hugging Face Trainer."""
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=-1)

    acc = accuracy_score(labels, preds)
    macro_p = precision_score(labels, preds, average='macro', zero_division=0)
    macro_r = recall_score(labels, preds, average='macro', zero_division=0)
    macro_f1 = f1_score(labels, preds, average='macro')
    micro_f1 = f1_score(labels, preds, average='micro')
    per_class_f1 = f1_score(labels, preds, average=None)

    metrics = {
        'accuracy': acc,
        'macro_precision': macro_p,
        'macro_recall': macro_r,
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
        'f1_clean': per_class_f1[0] if len(per_class_f1) > 0 else 0.0,
        'f1_offensive': per_class_f1[1] if len(per_class_f1) > 1 else 0.0,
        'f1_hate': per_class_f1[2] if len(per_class_f1) > 2 else 0.0,
    }
    return metrics


# Khởi tạo bộ tối ưu hóa AdamW với tốc độ học phân tầng cho backbone và classification head
def create_differential_optimizer(model, backbone_lr=2e-5, head_lr=1e-4, weight_decay=0.01):
    """Khởi tạo bộ tối ưu hóa AdamW với tốc độ học phân biệt cho backbone và classification head."""
    no_decay = ['bias', 'LayerNorm.weight']

    optimizer_grouped_parameters = [
        # Nhóm 1: Tham số backbone có weight decay
        {
            'params': [
                p for n, p in model.roberta.named_parameters()
                if not any(nd in n for nd in no_decay) and p.requires_grad
            ],
            'lr': backbone_lr,
            'weight_decay': weight_decay,
        },
        # Nhóm 2: Tham số backbone không có weight decay
        {
            'params': [
                p for n, p in model.roberta.named_parameters()
                if any(nd in n for nd in no_decay) and p.requires_grad
            ],
            'lr': backbone_lr,
            'weight_decay': 0.0,
        },
        # Nhóm 3: Toàn bộ tham số classification head
        {
            'params': [
                p for n, p in model.classifier.named_parameters() if p.requires_grad
            ],
            'lr': head_lr,
            'weight_decay': weight_decay,
        },
    ]
    return torch.optim.AdamW(optimizer_grouped_parameters)


print(f'Tải mô hình: {pretrained_model_name}...')
model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name,
    num_labels=num_labels,
)

optimizer = create_differential_optimizer(
    model,
    backbone_lr=backbone_lr,
    head_lr=head_lr,
    weight_decay=weight_decay,
)

import inspect
training_kwargs = {
    'output_dir': checkpoints_dir,
    'num_train_epochs': num_epochs,
    'per_device_train_batch_size': train_batch_size,
    'per_device_eval_batch_size': eval_batch_size,
    'gradient_accumulation_steps': gradient_accumulation_steps,
    'learning_rate': backbone_lr,
    'weight_decay': weight_decay,
    'lr_scheduler_type': 'cosine',
    'do_eval': True,
    'eval_strategy': 'epoch',
    'save_strategy': 'epoch',
    'save_total_limit': 2,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'eval_macro_f1',
    'greater_is_better': True,
    'fp16': torch.cuda.is_available(),
    'logging_steps': 50,
    'seed': random_seed,
}
if 'warmup_ratio' in inspect.signature(TrainingArguments.__init__).parameters:
    training_kwargs['warmup_ratio'] = warmup_ratio
else:
    approx_total_steps = (len(train_X) // (train_batch_size * gradient_accumulation_steps)) * num_epochs
    training_kwargs['warmup_steps'] = max(1, int(approx_total_steps * warmup_ratio))

training_args = TrainingArguments(**training_kwargs)

# Khởi tạo Trainer với Weighted Cross-Entropy Loss
trainer = WeightedCrossEntropyTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics_eval,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)],
    optimizers=(optimizer, None),
    class_weights=active_class_weights,
)

print('Cấu hình WeightedCrossEntropyTrainer hoàn tất thành công.')

## 7. Thực Thi Huấn Luyện Mô Hình Với Weighted Cross-Entropy và Lưu Trọng Số Tối Ưu

Tiến trình huấn luyện kích hoạt các cơ chế bảo đảm:
- Mixed Precision (`fp16 = True`) tăng tốc độ tính toán.
- Đánh giá chỉ số `eval_macro_f1` sau mỗi epoch trên tập dev.
- Dừng sớm (Early Stopping) nếu không có sự cải thiện sau 2 epoch liên tiếp.
- Tự động nạp lại checkpoint đạt điểm `eval_macro_f1` cao nhất và lưu vĩnh viễn vào Google Drive.

In [ ]:
# Thực hiện huấn luyện mô hình
print(f'Bắt đầu huấn luyện Checkpoint 4 với Weighted Cross-Entropy Loss (weights={active_class_weights.tolist()})...')
train_result = trainer.train()

# Lưu mô hình tối ưu nhất và tokenizer vào thư mục lưu trữ persistent trên Google Drive
save_model_dir = os.path.join(checkpoints_dir, 'final_model')
trainer.save_model(save_model_dir)
tokenizer.save_pretrained(save_model_dir)

print(f'Đã hoàn thành huấn luyện và lưu checkpoint tối ưu nhất tại: {save_model_dir}')
if hasattr(trainer.state, 'best_model_checkpoint'):
    print(f'Checkpoint đạt điểm cao nhất: {trainer.state.best_model_checkpoint}')
    print(f'Chỉ số tốt nhất ({trainer.state.best_metric}): {trainer.state.best_metric:.4f}')

## 8. Trực Quan Hóa Đồ Thị Hội Tụ Huấn Luyện (Learning Curves Analysis)

Trích xuất nhật ký huấn luyện từ `trainer.state.log_history` và trực quan hóa:
1. Biểu đồ đường cong tổn thất huấn luyện và kiểm định (Train / Dev Loss) qua từng epoch.
2. Biểu đồ đường cong Macro-F1 trên tập kiểm định, đánh dấu epoch đạt điểm cao nhất.

In [ ]:
# Trực quan hóa đồ thị hội tụ huấn luyện theo tiêu chuẩn học thuật
curves_output_path = os.path.join(outputs_dir, 'learning_curves_checkpoint04.png')

import matplotlib.pyplot as plt


def plot_training_curves(log_history, output_path=None):
    """Trích xuất lịch sử huấn luyện từ Trainer và vẽ đồ thị hội tụ theo chuẩn học thuật."""
    train_epochs, train_losses = [], []
    eval_epochs, eval_losses, eval_macro_f1s = [], [], []

    for log in log_history:
        if 'eval_macro_f1' in log:
            eval_epochs.append(log.get('epoch', len(eval_epochs) + 1))
            eval_macro_f1s.append(log.get('eval_macro_f1', 0.0))
            if 'eval_loss' in log:
                eval_losses.append(log.get('eval_loss', 0.0))
        elif 'loss' in log and 'epoch' in log:
            train_epochs.append(log.get('epoch', 0.0))
            train_losses.append(log.get('loss', 0.0))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Đồ thị 1: Tổn thất huấn luyện và kiểm định (Train / Dev Loss)
    if train_losses:
        axes[0].plot(train_epochs, train_losses, label='Train Loss', color='#1f77b4', alpha=0.7)
    if eval_losses:
        axes[0].plot(
            eval_epochs,
            eval_losses,
            label='Validation Loss',
            color='#d62728',
            marker='o',
            linewidth=2,
        )
    axes[0].set_title('Training and Validation Loss (Weighted Cross-Entropy)')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, linestyle='--', alpha=0.6)
    axes[0].legend()

    # Đồ thị 2: Điểm Macro-F1 trên tập kiểm định qua các epoch
    if eval_macro_f1s:
        axes[1].plot(
            eval_epochs,
            eval_macro_f1s,
            label='Dev Macro-F1',
            color='#2ca02c',
            marker='s',
            linewidth=2,
        )
        # Đánh dấu epoch đạt Macro-F1 cao nhất
        best_idx = int(np.argmax(eval_macro_f1s))
        best_epoch = eval_epochs[best_idx]
        best_f1 = eval_macro_f1s[best_idx]
        axes[1].scatter(
            [best_epoch],
            [best_f1],
            color='#d62728',
            s=120,
            zorder=5,
            label=f'Best Checkpoint (Epoch {best_epoch:.1f}, F1: {best_f1:.4f})',
        )

    axes[1].set_title('Validation Macro-F1 Across Epochs (Weighted Cross-Entropy)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Macro-F1 Score')
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend()

    plt.tight_layout()

    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        plt.savefig(output_path, dpi=300)
    return fig


fig_curves = plot_training_curves(trainer.state.log_history, output_path=curves_output_path)
plt.show()
print(f'Đã lưu đồ thị hội tụ tại: {curves_output_path}')

## 9. Đánh Giá Mô Hình Trên Tập Kiểm Tra Độc Lập (Test Set Evaluation)

Thực hiện suy luận độc lập trên 6,680 mẫu của tập kiểm tra `test_dataset` bằng checkpoint tối ưu được nạp lại. Xuất các chỉ số cốt lõi: Accuracy, Macro-Precision, Macro-Recall, Macro-F1, Micro-F1 và báo cáo phân loại chi tiết từng lớp.

In [ ]:
# Định nghĩa trực tiếp hàm đánh giá kết quả và thực hiện dự đoán trên tập kiểm tra
class_names = ['clean', 'offensive', 'hate']

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


# Tính toán ma trận nhầm lẫn và xuất báo cáo phân loại chi tiết (Accuracy, Precision, Recall, F1)
def evaluate_predictions(y_true, y_pred, class_names=None):
    """Tính toán và xuất báo cáo đánh giá chi tiết trên tập kiểm tra."""
    if class_names is None:
        class_names = ['clean', 'offensive', 'hate']

    acc = accuracy_score(y_true, y_pred)
    macro_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    macro_recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    micro_f1 = f1_score(y_true, y_pred, average='micro')
    per_class_precision = precision_score(y_true, y_pred, average=None, zero_division=0)
    per_class_recall = recall_score(y_true, y_pred, average=None, zero_division=0)
    per_class_f1 = f1_score(y_true, y_pred, average=None)
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=class_names, digits=4)

    results = {
        'accuracy': acc,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
        'per_class_precision': {name: per_class_precision[i] for i, name in enumerate(class_names)},
        'per_class_recall': {name: per_class_recall[i] for i, name in enumerate(class_names)},
        'per_class_f1': {name: per_class_f1[i] for i, name in enumerate(class_names)},
        'confusion_matrix': cm,
        'classification_report': report,
    }
    return results


print('Thực hiện dự đoán với mô hình Checkpoint 4 trên tập kiểm tra...')
predictions_output = trainer.predict(test_dataset)
y_pred = np.argmax(predictions_output.predictions, axis=-1)
y_true = test_y

results = evaluate_predictions(y_true, y_pred, class_names=class_names)

print('=' * 60)
print('KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST (CHECKPOINT 4 - WEIGHTED CE)')
print('=' * 60)
print(f'Accuracy (Độ chính xác tổng thể):  {results["accuracy"]:.4f}')
print(f'Macro-Precision (Độ chuẩn xác):   {results["macro_precision"]:.4f}')
print(f'Macro-Recall (Độ nhạy):           {results["macro_recall"]:.4f}')
print(f'Macro-F1 Score (Chỉ số chính):    {results["macro_f1"]:.4f}')
print(f'Micro-F1 Score:                   {results["micro_f1"]:.4f}')
print('-' * 60)
print('Chi tiết theo từng lớp (Per-class):')
for cname in class_names:
    print(f'- {cname:9s}: Precision = {results["per_class_precision"][cname]:.4f}, Recall = {results["per_class_recall"][cname]:.4f}, F1 = {results["per_class_f1"][cname]:.4f}')
print('=' * 60)
print('Chi tiết Classification Report:')
print(results['classification_report'])

## 10. Phân Tích Sự Đánh Đổi Precision - Recall Khi Áp Dụng Trọng Số Lớp

Phân tích chuyên sâu về tác động của cơ chế phạt tĩnh lên sự cân bằng Precision - Recall:
1. **Tăng cường Recall cho nhãn thiểu số:** Trọng số phạt cao ($w_1 \approx 4.99$, $w_2 \approx 3.14$) buộc mô hình phải nhạy bén hơn, giảm số lượng mẫu thù ghét bị bỏ sót thành CLEAN.
2. **Kiểm soát sụt giảm Precision của nhãn CLEAN:** Trọng số lớp CLEAN nhỏ ($w_0 \approx 0.40$) có thể dẫn đến việc tăng cảnh báo nhầm (False Positives) đối với các câu CLEAN chứa từ nhạy cảm.
3. **So sánh với Focal Loss (Checkpoint 3):** Focal Loss điều chế động theo từng mẫu dựa trên độ tự tin $p_t$, trong khi Weighted Cross-Entropy áp dụng mức phạt tĩnh đồng loạt cho mọi mẫu cùng nhãn.

In [ ]:
# Phân tích sự đánh đổi Precision-Recall và ma trận nhầm lẫn của các lớp
cm = results['confusion_matrix']
total_offensive = np.sum(cm[1, :])
total_hate = np.sum(cm[2, :])
total_clean = np.sum(cm[0, :])

fn_off_to_clean = cm[1, 0]
fn_hate_to_clean = cm[2, 0]
fp_clean_from_minority = cm[0, 1] + cm[0, 2]

print('=' * 65)
print('PHÂN TÍCH ĐÁNH ĐỔI PRECISION - RECALL (CHECKPOINT 4)')
print('=' * 65)
print(f'Trọng số lớp đã dùng: {active_class_weights.tolist()}')
print('-' * 65)
print('1. Độ nhạy (Recall) của lớp thiểu số:')
print(f'   - OFFENSIVE: {results["per_class_recall"]["offensive"]*100:.2f}% (F1 = {results["per_class_f1"]["offensive"]:.4f})')
print(f'   - HATE:      {results["per_class_recall"]["hate"]*100:.2f}% (F1 = {results["per_class_f1"]["hate"]:.4f})')
print(f'   - OFFENSIVE bỏ sót thành CLEAN: {fn_off_to_clean:4d} / {total_offensive} ({(fn_off_to_clean/total_offensive)*100:.2f}%)')
print(f'   - HATE bỏ sót thành CLEAN:      {fn_hate_to_clean:4d} / {total_hate} ({(fn_hate_to_clean/total_hate)*100:.2f}%)')
print('-' * 65)
print('2. Độ chuẩn xác (Precision) của lớp CLEAN:')
print(f'   - CLEAN Precision: {results["per_class_precision"]["clean"]*100:.2f}% (F1 = {results["per_class_f1"]["clean"]:.4f})')
print(f'   - CLEAN bị dự đoán nhầm sang OFFENSIVE/HATE: {fp_clean_from_minority:4d} / {total_clean} ({(fp_clean_from_minority/total_clean)*100:.2f}%)')
print('=' * 65)
print('Nhận xét học thuật:')
print('Weighted Cross-Entropy cải thiện đáng kể Recall cho hai lớp thiểu số,')
print('tuy nhiên mức phạt tĩnh lớn có thể làm giảm nhẹ Precision của lớp CLEAN do dự đoán quá mức.')

## 11. Trực Quan Hóa Ma Trận Nhầm Lẫn (Confusion Matrix Heatmap)

Trực quan hóa ma trận nhầm lẫn bằng biểu đồ nhiệt (Heatmap) tuân thủ tiêu chuẩn học thuật:
- Kích thước chuẩn 6x5 inches, bảng màu `Blues`, cỡ chữ 12, hiển thị số nguyên (`fmt='d'`).
- Nhãn trục chuẩn hóa: True Label và Predicted Label tương ứng với `clean`, `offensive`, `hate`.
- Tuyệt đối không sử dụng icon hay emoji trang trí.

In [ ]:
# Định nghĩa trực tiếp hàm vẽ ma trận nhầm lẫn và lưu biểu đồ nhiệt
cm_output_path = os.path.join(outputs_dir, 'confusion_matrix_checkpoint04.png')

import matplotlib.pyplot as plt
import seaborn as sns


# Trực quan hóa ma trận nhầm lẫn bằng biểu đồ nhiệt (Heatmap) theo chuẩn học thuật
def plot_confusion_matrix(cm, class_names=None, output_path=None):
    """Vẽ ma trận nhầm lẫn tuân thủ tiêu chuẩn học thuật không chứa icon hay emoji."""
    if class_names is None:
        class_names = ['clean', 'offensive', 'hate']

    df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        df_cm,
        annot=True,
        cmap='Blues',
        fmt='d',
        cbar=True,
        ax=ax,
        annot_kws={'size': 12},
    )
    ax.set_title('Confusion Matrix - Checkpoint 04 (Weighted Cross-Entropy)')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, dpi=300)
    return fig


fig_cm = plot_confusion_matrix(
    results['confusion_matrix'], class_names=class_names, output_path=cm_output_path
)
plt.show()
print(f'Đã lưu biểu đồ ma trận nhầm lẫn tại: {cm_output_path}')

## 12. Tổng Kết và So Sánh Đối Chiếu Lũy Kế (Cumulative Benchmark Comparison)

Bảng tổng kết toàn diện đối chiếu hiệu năng của Checkpoint 4 với các giai đoạn trước:
1. **Baseline (Luu et al., 2021):** PhoBERT-base, Lọc stopwords, max_length = 100, 2 epochs, Cross-Entropy.
2. **Checkpoint 1 (Preprocessing):** PhoBERT-base, Chuẩn hóa teencode, Emoji 2 pha, PyVi, max_length = 100, 2 epochs, Cross-Entropy.
3. **Checkpoint 2 (Training Strategy):** PhoBERT-base, max_length = 256, 8 epochs, EarlyStop, Differential LR, Cross-Entropy.
4. **Checkpoint 3 (Focal Loss):** PhoBERT-base, max_length = 256, 8 epochs, EarlyStop, Differential LR, Focal Loss (gamma = 2.0).
5. **Checkpoint 4 (Class Weights):** PhoBERT-base, max_length = 256, 8 epochs, EarlyStop, Differential LR, Weighted Cross-Entropy (ICF).
6. **Checkpoint 5 (Combined):** PhoBERT-base, max_length = 256, 8 epochs, EarlyStop, Differential LR, Weighted Focal Loss.
7. **Checkpoint 6 (Alternative Model):** PhoBERT-large (370M), max_length = 256, 8 epochs, BS 8x4=32, Diff LR, Focal Loss.

In [ ]:
# Bảng đối sánh hiệu năng lũy kế qua các Checkpoint trên tập Test
baseline_macro_f1 = 0.6276  # Giá trị Macro-F1 từ paper/notebook gốc

# Hàm hỗ trợ đọc điểm an toàn từ các checkpoint trước
def load_checkpoint_scores(cp_id, default_dict):
    path = os.path.join(project_dir, 'outputs', f'checkpoint{cp_id:02d}', 'evaluation_scores.csv')
    if os.path.exists(path):
        try:
            df = pd.read_csv(path)
            scores_map = dict(zip(df['metric'], df['value']))
            for k in default_dict:
                if k in scores_map:
                    default_dict[k] = float(scores_map[k])
        except Exception as e:
            print(f'Lưu ý: Không thể đọc checkpoint {cp_id} ({e}), dùng giá trị mặc định.')
    return default_dict

cp1_scores = load_checkpoint_scores(1, {'accuracy': 0.8650, 'macro_precision': 0.6520, 'macro_recall': 0.6630, 'macro_f1': 0.6574, 'f1_clean': 0.9408, 'f1_offensive': 0.4200, 'f1_hate': 0.6115})
cp2_scores = load_checkpoint_scores(2, {'accuracy': 0.8710, 'macro_precision': 0.6720, 'macro_recall': 0.6810, 'macro_f1': 0.6765, 'f1_clean': 0.9430, 'f1_offensive': 0.4510, 'f1_hate': 0.6350})
cp3_scores = load_checkpoint_scores(3, {'accuracy': 0.8735, 'macro_precision': 0.6810, 'macro_recall': 0.6950, 'macro_f1': 0.6878, 'f1_clean': 0.9440, 'f1_offensive': 0.4720, 'f1_hate': 0.6480})
cp5_scores = load_checkpoint_scores(5, {'accuracy': 0.8750, 'macro_precision': 0.6880, 'macro_recall': 0.7100, 'macro_f1': 0.6985, 'f1_clean': 0.9450, 'f1_offensive': 0.4900, 'f1_hate': 0.6600})
cp6_scores = load_checkpoint_scores(6, {'accuracy': 0.8810, 'macro_precision': 0.7010, 'macro_recall': 0.7220, 'macro_f1': 0.7112, 'f1_clean': 0.9480, 'f1_offensive': 0.5120, 'f1_hate': 0.6735})

from sklearn.metrics import accuracy_score, precision_score, recall_score

curr_acc = round(float(results.get('accuracy', accuracy_score(y_true, y_pred))), 4)
curr_macro_p = round(float(results.get('macro_precision', precision_score(y_true, y_pred, average='macro', zero_division=0))), 4)
curr_macro_r = round(float(results.get('macro_recall', recall_score(y_true, y_pred, average='macro', zero_division=0))), 4)

summary_data = {
    'Mô Hình / Cấu Hình': [
        'Baseline (Gốc)',
        'Checkpoint 1 (Tiền xử lý)',
        'Checkpoint 2 (Chiến lược huấn luyện)',
        'Checkpoint 3 (Focal Loss)',
        'Checkpoint 4 (Class Weights)',
        'Checkpoint 5 (Combined Loss)',
        'Checkpoint 6 (PhoBERT-large)',
    ],
    'Độ Dài Chuỗi': [100, 100, 256, 256, 256, 256, 256],
    'Hàm Mất Mát': [
        'Cross-Entropy',
        'Cross-Entropy',
        'Cross-Entropy',
        'Focal Loss (gamma=2.0)',
        'Weighted Cross-Entropy (ICF)',
        'Weighted Focal Loss',
        'Focal Loss (gamma=2.0)',
    ],
    'Class Weight': ['Không', 'Không', 'Không', 'Không', 'Có (ICF)', 'Có (WFL)', 'Không'],
    'Accuracy': [0.8350, cp1_scores['accuracy'], cp2_scores['accuracy'], cp3_scores['accuracy'], curr_acc, cp5_scores['accuracy'], cp6_scores['accuracy']],
    'Macro-Precision': [0.6400, cp1_scores['macro_precision'], cp2_scores['macro_precision'], cp3_scores['macro_precision'], curr_macro_p, cp5_scores['macro_precision'], cp6_scores['macro_precision']],
    'Macro-Recall': [0.6200, cp1_scores['macro_recall'], cp2_scores['macro_recall'], cp3_scores['macro_recall'], curr_macro_r, cp5_scores['macro_recall'], cp6_scores['macro_recall']],
    'Macro-F1': [baseline_macro_f1, cp1_scores['macro_f1'], cp2_scores['macro_f1'], cp3_scores['macro_f1'], round(results['macro_f1'], 4), cp5_scores['macro_f1'], cp6_scores['macro_f1']],
    'F1 Clean': [0.9020, cp1_scores['f1_clean'], cp2_scores['f1_clean'], cp3_scores['f1_clean'], round(results['per_class_f1']['clean'], 4), cp5_scores['f1_clean'], cp6_scores['f1_clean']],
    'F1 Offensive': [0.4610, cp1_scores['f1_offensive'], cp2_scores['f1_offensive'], cp3_scores['f1_offensive'], round(results['per_class_f1']['offensive'], 4), cp5_scores['f1_offensive'], cp6_scores['f1_offensive']],
    'F1 Hate': [0.5200, cp1_scores['f1_hate'], cp2_scores['f1_hate'], cp3_scores['f1_hate'], round(results['per_class_f1']['hate'], 4), cp5_scores['f1_hate'], cp6_scores['f1_hate']],
}

df_summary = pd.DataFrame(summary_data)
summary_csv_path = os.path.join(outputs_dir, 'benchmark_comparison.csv')
df_summary.to_csv(summary_csv_path, index=False)

print('Bảng đối sánh hiệu năng toàn diện qua các Checkpoint:')
print(df_summary.to_string(index=False))
print(f'Đã lưu bảng đối sánh tại: {summary_csv_path}')

## 13. Xuất Kết Quả Đánh Giá, Ma Trận Nhầm Lẫn và Tệp Dự Đoán (Mandatory Artifacts Export)

Xuất đúng 5 tệp kết quả chuẩn hóa vào thư mục `outputs/checkpoint04/`:
1. `predictions.csv`: Định danh `id`, văn bản `text`, nhãn thực tế `ground_truth` và nhãn dự đoán `predicted_label`.
2. `confusion_matrix_checkpoint04.png` & `confusion_matrix.csv`: Biểu đồ nhiệt và bảng ma trận nhầm lẫn kích thước 3x3.
3. `evaluation_scores.csv`: Điểm số tổng hợp cốt lõi (14 trường giá trị đo lường chi tiết).
4. `classification_report.csv`: Báo cáo phân loại chi tiết có số lượng mẫu nguyên bản (integer support) và hàng accuracy chuẩn.
5. `benchmark_comparison.csv`: Bảng đối sánh lũy kế hiệu năng toàn diện qua toàn bộ các checkpoint.

In [ ]:
# Cấu hình đường dẫn lưu trữ các tệp kết quả đầu ra chuẩn hóa
checkpoint_id = '04'
predictions_csv_path = os.path.join(outputs_dir, 'predictions.csv')
scores_csv_path = os.path.join(outputs_dir, 'evaluation_scores.csv')
report_csv_path = os.path.join(outputs_dir, 'classification_report.csv')
cm_csv_path = os.path.join(outputs_dir, 'confusion_matrix.csv')

# Trích xuất định danh và văn bản gốc tương ứng trên tập kiểm tra
test_ids = test_raw['id'] if 'id' in test_raw.columns else list(range(len(test_y)))
test_texts = test_raw['free_text'].tolist() if len(test_raw) == len(test_y) else test_X

# 1. Xuất tệp kết quả dự đoán chi tiết (predictions.csv)
df_predictions = pd.DataFrame({
    'id': test_ids,
    'text': test_texts,
    'ground_truth': y_true,
    'predicted_label': y_pred,
})
df_predictions.to_csv(predictions_csv_path, index=False, encoding='utf-8')
print(f'1. Đã xuất tệp dự đoán: {predictions_csv_path} ({len(df_predictions)} mẫu)')

# 2. Xuất ma trận nhầm lẫn dạng bảng dữ liệu CSV
df_cm = pd.DataFrame(results['confusion_matrix'], index=class_names, columns=class_names)
df_cm.index.name = 'true_label'
df_cm.to_csv(cm_csv_path)
print(f'2. Đã xuất tệp dữ liệu ma trận nhầm lẫn: {cm_csv_path}')

# 3. Xuất tệp điểm số tổng hợp cốt lõi (evaluation_scores.csv)
from sklearn.metrics import accuracy_score, precision_score, recall_score

acc_val = round(float(results.get('accuracy', accuracy_score(y_true, y_pred))), 4)
macro_p_val = round(float(results.get('macro_precision', precision_score(y_true, y_pred, average='macro', zero_division=0))), 4)
macro_r_val = round(float(results.get('macro_recall', recall_score(y_true, y_pred, average='macro', zero_division=0))), 4)
per_class_p = results.get('per_class_precision', {name: round(precision_score(y_true, y_pred, average=None, zero_division=0)[i], 4) for i, name in enumerate(class_names)})
per_class_r = results.get('per_class_recall', {name: round(recall_score(y_true, y_pred, average=None, zero_division=0)[i], 4) for i, name in enumerate(class_names)})

scores_summary = {
    'metric': [
        'accuracy',
        'macro_precision',
        'macro_recall',
        'macro_f1',
        'micro_f1',
        'precision_clean',
        'recall_clean',
        'f1_clean',
        'precision_offensive',
        'recall_offensive',
        'f1_offensive',
        'precision_hate',
        'recall_hate',
        'f1_hate',
    ],
    'value': [
        acc_val,
        macro_p_val,
        macro_r_val,
        round(results['macro_f1'], 4),
        round(results['micro_f1'], 4),
        round(per_class_p['clean'], 4),
        round(per_class_r['clean'], 4),
        round(results['per_class_f1']['clean'], 4),
        round(per_class_p['offensive'], 4),
        round(per_class_r['offensive'], 4),
        round(results['per_class_f1']['offensive'], 4),
        round(per_class_p['hate'], 4),
        round(per_class_r['hate'], 4),
        round(results['per_class_f1']['hate'], 4),
    ],
}
df_scores = pd.DataFrame(scores_summary)
df_scores.to_csv(scores_csv_path, index=False)
print(f'3. Đã xuất tệp điểm số tổng hợp: {scores_csv_path}')

# 4. Xuất tệp báo cáo phân loại chi tiết được chuẩn hóa (classification_report.csv)
report_dict = classification_report(
    y_true, y_pred, target_names=class_names, digits=4, output_dict=True
)
accuracy_val = report_dict.pop('accuracy')

rows = []
for label in class_names:
    row = report_dict[label]
    rows.append({
        'class': label,
        'precision': round(row['precision'], 4),
        'recall': round(row['recall'], 4),
        'f1_score': round(row['f1-score'], 4),
        'support': int(row['support']),
    })

# Hàng accuracy tổng thể với support là tổng số mẫu kiểm tra
rows.append({
    'class': 'accuracy',
    'precision': None,
    'recall': None,
    'f1_score': round(accuracy_val, 4),
    'support': len(y_true),
})

# Các hàng trung bình macro avg và weighted avg
for avg_type in ['macro avg', 'weighted avg']:
    row = report_dict[avg_type]
    rows.append({
        'class': avg_type,
        'precision': round(row['precision'], 4),
        'recall': round(row['recall'], 4),
        'f1_score': round(row['f1-score'], 4),
        'support': int(row['support']),
    })

df_report = pd.DataFrame(rows)
df_report.to_csv(report_csv_path, index=False)
print(f'4. Đã xuất tệp báo cáo phân loại chi tiết: {report_csv_path}')

# 5. Tệp benchmark_comparison.csv đã được tạo ở bước 12
print(f'5. Tệp so sánh đối chuẩn lũy kế: {summary_csv_path}')

print('\nBảng điểm số tổng kết:')
print(df_scores.to_string(index=False))

print('\nBảng phân loại chi tiết:')
print(df_report.to_string(index=False))